# F1-M02: Limpieza de Datos

**TFM: Predicción de Abandono Universitario**

| | |
|---|---|
| **Autora** | María José Morte |
| **Email** | mjmorteruiz@uoc.edu (UOC) \| morte@uji.es (UJI) |

---

## 🎯 ¿Qué hace este notebook?

Limpia y estandariza las **9 tablas** de datos originales de la UJI.
Los datos vienen del sistema de gestión universitario y tienen nombres
de columnas en valenciano, valores vacíos como `"-"`, y tipos de datos
inconsistentes. Este notebook los deja listos para trabajar.

Las transformaciones que aplica son:

1. **Renombrar columnas** a snake_case (y traducir del valenciano al español)
2. **Reemplazar valores vacíos** (`"-"`, `"N/A"`, `""`, etc.) por `NaN`
3. **Estandarizar tipos** de datos (notas a numérico, fechas a datetime, etc.)
4. **Guardar** cada tabla limpia como parquet en `data/01_interim/`
5. **Generar** página HTML interactiva con los detalles de cada limpieza

## ⚠️ Requisitos

- Haber ejecutado `00_configuracion_proyecto.ipynb` (crea las carpetas)
- Los 2 archivos Excel originales en `data/00_raw/`

## 📦 ¿Qué genera?

| Archivo | Ubicación | Descripción |
|---|---|---|
| 9 parquets limpios | `data/01_interim/` | Uno por cada tabla (expedientes.parquet, etc.) |
| m02_limpieza.html | `docs/html/fase1/` | Página interactiva con detalle de transformaciones |

## 📁 Flujo de datos

```
data/00_raw/*.xlsx  →  limpieza  →  data/01_interim/*.parquet
```

## 💡 ¿Por qué valenciano?

Los datos de la UJI (Universitat Jaume I) tienen columnas en valenciano
(`Curs_Aca`, `Egressat`, `Nom_Beca`, etc.). Las traducimos al español
para que el análisis sea más legible.

## 📌 Siguiente

- `f1_m03_reportes_clean.ipynb` (reportes Sweetviz de datos limpios)

In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN
# ============================================================================
#
# Esta celda hace 3 cosas:
#   1. Importa las librerías necesarias (pandas, numpy, re)
#   2. Detecta el entorno (Colab o Local) y encuentra la raíz del proyecto
#   3. Importa rutas, constantes y funciones de src.config
#
# VALORES_VACIOS viene de src.config (definido en config_datos.py).
# Es la lista de strings que se consideran "vacíos" y se reemplazan por NaN.
# ============================================================================

import sys
import warnings
import re
from pathlib import Path

warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

# --- Detectar entorno ---
# En Colab: monta Google Drive y usa la ruta fija de AU_UJI
# En Local: sube niveles desde la carpeta actual hasta encontrar 'src/'
#           Esto funciona desde cualquier carpeta dentro del proyecto.
ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists():
        break
    ROOT = ROOT.parent

# Verificar que encontramos la raíz del proyecto
if not (ROOT / 'src').exists():
    raise FileNotFoundError(
        f'No se encontró la carpeta src/ en {ROOT}\n'
        f'Asegúrate de que este notebook está dentro del proyecto (AU_UJI/)'
    )

sys.path.insert(0, str(ROOT))

# --- Imports del proyecto ---
# VALORES_VACIOS: lista de strings que se consideran "vacíos" en los Excel
#   Ejemplo: ['-', '', ' ', 'N/A', 'n/a', 'NA', 'null', 'NULL', 'None']
# MAPEO_HOJAS: diccionario que traduce nombres de hojas del Excel a nombres internos
#   Ejemplo: {'Nac-Sexo_Nacionalidad': 'demograficos', 'Circunstancias': 'becas'}
from src.config import (
    RUTA_RAW, RUTA_INTERIM, RUTA_HTML,
    EXCEL_PRINCIPAL, EXCEL_PREINSCRIPCION,
    MAPEO_HOJAS, VALORES_VACIOS,
    info_entorno
)

# Funciones de utilidad
from src.utils import crear_directorios, formato_numero_es

# Funciones para generar HTML
from src.html import (
    generar_html_navegacion_completa,
    guardar_html
)
from src.html.render import render_base_html

# --- Crear carpetas de salida ---
# RUTA_INTERIM: donde se guardan los parquets limpios (data/01_interim/)
# RUTA_FASE1: donde se guarda el HTML de esta fase (docs/html/fase1/)
RUTA_FASE1 = RUTA_HTML / 'fase1'
crear_directorios([RUTA_INTERIM, RUTA_FASE1])

# Atajo para formatear números en español (1.234.567 en vez de 1,234,567)
fmt = formato_numero_es

# Mostrar información del entorno
info_entorno()

✓ Directorios verificados: 2
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓

In [2]:
# ============================================================================
# CELDA 2: FUNCIONES DE LIMPIEZA
# ============================================================================
#
# Define 2 funciones que se usan en la celda siguiente:
#
# 1. normalizar_nombre_columna(nombre)
#    Convierte nombres de columnas del Excel a snake_case.
#    Primero busca en un mapeo manual (valenciano → español),
#    y si no lo encuentra, aplica conversión automática.
#
#    Ejemplo:
#      'Curs_Aca'        → 'curso_aca'       (mapeo manual)
#      'Nom_Beca'        → 'nombre_beca'     (mapeo manual)
#      'CualquierNombre' → 'cualquier_nombre' (automático)
#
# 2. limpiar_dataframe(df, nombre_tabla)
#    Aplica la limpieza estándar a un DataFrame:
#    - Renombra columnas con la función anterior
#    - Reemplaza valores vacíos (de VALORES_VACIOS) por NaN
#    - Convierte notas a numérico (solo en tabla expedientes)
#
# ¿Por qué un mapeo manual?
#    Los datos de la UJI usan nombres en valenciano que no se pueden
#    traducir automáticamente. Ejemplo: 'Egressat' → 'egresado',
#    'Cred_Superats' → 'cred_superados'. Sin el mapeo, quedarían
#    en valenciano y el análisis sería confuso.
# ============================================================================

def normalizar_nombre_columna(nombre):
    """
    Convierte nombre de columna a snake_case español.
    
    Primero busca en el mapeo manual (valenciano → español).
    Si no lo encuentra, aplica conversión automática a snake_case.
    
    Parameters
    ----------
    nombre : str
        Nombre original de la columna del Excel
        
    Returns
    -------
    str
        Nombre normalizado en snake_case español
    """
    # --- Mapeo manual: valenciano → español ---
    # Estos nombres no se pueden traducir automáticamente
    mapeo = {
        # Campos temporales
        'Curs_Aca': 'curso_aca',
        'Curs_Aca_Ini': 'curso_aca_ini',
        'Curs_Aca_Fi': 'curso_aca_fin',
        'Mat_Curs_Aca': 'mat_curso_aca',
        # Identificadores
        'Exp_Tit_Id': 'exp_tit_id',
        'Per_Id_Fictici': 'per_id_ficticio',
        # Titulaciones
        'Titulacio': 'titulacion',
        'Cred_Titulacio': 'cred_titulacion',
        'Cred_Matriculats': 'cred_matriculados',
        'Cred_Superats': 'cred_superados',
        'Tipus': 'tipo',
        'Tipus_Estudi': 'tipo_estudio',
        # Demográficos
        'Data_Naixement': 'fecha_nacimiento',
        'Id_Pais': 'id_pais',
        'Pais_Nom': 'pais_nombre',
        'Sexe': 'sexo',
        # Domicilio
        'Poblacio': 'poblacion',
        'Provincia': 'provincia',
        'Pais': 'pais',
        'Tipus_Domicili': 'tipo_domicilio',
        # Becas y trabajo
        'Nom_Beca': 'nombre_beca',
        'Nom_Treball': 'nombre_trabajo',
        # Notas y medias
        'Mitjana_Titulacio_Curs': 'media_titulacion_curso',
        'Mitjana_Titulacio_Alumne': 'media_titulacion_alumno',
        'Nota_Selectivitat': 'nota_selectividad',
        'Nota_Acces': 'nota_acceso',
        'Mitjana_Curs': 'media_curso',
        # Pagos
        'Forma_Pagament': 'forma_de_pago',
        'Num_Pagaments': 'numero_pagos',
        # Estado académico
        'Egressat': 'egresado',
        'Nou': 'nuevo',
    }

    # Si está en el mapeo, devolver la traducción directa
    if nombre in mapeo:
        return mapeo[nombre]

    # Si no, convertir automáticamente a snake_case:
    # 'CualquierNombre' → 'cualquier_nombre'
    # Paso 1: insertar '_' antes de mayúsculas que siguen a minúsculas
    s = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', nombre)
    # Paso 2: insertar '_' entre minúscula/número y mayúscula
    s = re.sub('([a-z0-9])([A-Z])', r'\1_\2', s)
    # Paso 3: todo a minúsculas y limpiar espacios
    return s.lower().replace(' ', '_').replace('__', '_')


def limpiar_dataframe(df, nombre_tabla):
    """
    Aplica limpieza estándar a un DataFrame.
    
    Pasos:
      1. Renombrar columnas a snake_case español
      2. Reemplazar valores vacíos por NaN
      3. Conversiones de tipo específicas por tabla
    
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame con datos crudos del Excel
    nombre_tabla : str
        Nombre interno de la tabla ('expedientes', 'notas', etc.)
    
    Returns
    -------
    tuple
        (df_limpio, columnas_antes, columnas_despues)
    """
    df_clean = df.copy()

    # 1. Guardar nombres originales (para la documentación HTML)
    cols_antes = df_clean.columns.tolist()

    # 2. Renombrar columnas: valenciano → snake_case español
    df_clean.columns = [normalizar_nombre_columna(c) for c in df_clean.columns]
    cols_despues = df_clean.columns.tolist()

    # 3. Reemplazar valores vacíos por NaN
    #    VALORES_VACIOS viene de src.config: ['-', '', ' ', 'N/A', 'null', ...]
    df_clean = df_clean.replace(VALORES_VACIOS, np.nan)

    # 4. Limpiezas específicas por tabla
    #    En la tabla 'expedientes', las notas vienen como texto en el Excel
    #    (ejemplo: '7,5' o 'N/A'). Las convertimos a float.
    if nombre_tabla == 'expedientes':
        for col in ['nota_selectividad', 'nota_acceso', 'media_curso']:
            if col in df_clean.columns:
                df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

    return df_clean, cols_antes, cols_despues


print('✅ Funciones de limpieza definidas')

✅ Funciones de limpieza definidas


In [3]:
# ============================================================================
# CELDA 3: CARGAR Y LIMPIAR DATOS
# ============================================================================
#
# Para cada tabla del Excel:
#   1. Lee la hoja con pd.read_excel (datos crudos)
#   2. Aplica limpiar_dataframe() (renombrar, NaN, tipos)
#   3. Guarda el resultado como parquet en data/01_interim/
#   4. Registra las métricas (filas, columnas, antes/después)
#
# ¿Por qué parquet?
#   - Es ~10x más rápido de leer/escribir que Excel
#   - Mantiene los tipos de datos (Excel pierde información)
#   - Ocupa ~5x menos espacio
#   - Es el formato estándar en ciencia de datos
#
# tablas_info: diccionario con las métricas de cada tabla limpia.
# Se usa después para generar el HTML con los detalles.
# ============================================================================

print('=' * 60)
print('CARGANDO Y LIMPIANDO DATOS')
print('=' * 60)

# Diccionario donde guardamos las métricas de cada tabla procesada
tablas_info = {}

# --- Excel Principal (8 hojas) ---
# MAPEO_HOJAS traduce nombre de hoja → nombre interno
# Ejemplo: 'Nac-Sexo_Nacionalidad' → 'demograficos'
print(f'\n📖 Leyendo {EXCEL_PRINCIPAL.name}...')
for hoja_excel, nombre_interno in MAPEO_HOJAS.items():
    try:
        # Leer hoja cruda del Excel
        df_raw = pd.read_excel(EXCEL_PRINCIPAL, sheet_name=hoja_excel)

        # Aplicar limpieza: renombrar columnas, NaN, tipos
        df_clean, cols_antes, cols_despues = limpiar_dataframe(df_raw, nombre_interno)

        # Guardar como parquet (mucho más rápido que Excel para los módulos siguientes)
        ruta_parquet = RUTA_INTERIM / f'{nombre_interno}.parquet'
        df_clean.to_parquet(ruta_parquet, index=False)

        # Registrar métricas para el HTML
        tablas_info[nombre_interno] = {
            'filas': len(df_clean),
            'columnas': len(df_clean.columns),
            'cols_antes': cols_antes,     # nombres originales (valenciano)
            'cols_despues': cols_despues   # nombres limpios (español)
        }
        print(f'   ✅ {nombre_interno:15s}: {len(df_clean):>8,} filas × {len(df_clean.columns)} cols → {ruta_parquet.name}')
    except Exception as e:
        print(f'   ❌ Error en {hoja_excel}: {e}')

# --- Excel Preinscripción (1 hoja) ---
# Este archivo tiene una sola hoja con datos de preinscripción
print(f'\n📖 Leyendo {EXCEL_PREINSCRIPCION.name}...')
try:
    df_raw = pd.read_excel(EXCEL_PREINSCRIPCION)
    df_clean, cols_antes, cols_despues = limpiar_dataframe(df_raw, 'preinscripcion')

    ruta_parquet = RUTA_INTERIM / 'preinscripcion.parquet'
    df_clean.to_parquet(ruta_parquet, index=False)

    tablas_info['preinscripcion'] = {
        'filas': len(df_clean),
        'columnas': len(df_clean.columns),
        'cols_antes': cols_antes,
        'cols_despues': cols_despues
    }
    print(f'   ✅ {"preinscripcion":15s}: {len(df_clean):>8,} filas × {len(df_clean.columns)} cols → {ruta_parquet.name}')
except Exception as e:
    print(f'   ❌ Error: {e}')

print(f'\n✅ {len(tablas_info)} tablas procesadas y guardadas en {RUTA_INTERIM}')

CARGANDO Y LIMPIANDO DATOS

📖 Leyendo datos_proyecto_sin_preinscrip.xlsx...


   ✅ titulaciones   :       45 filas × 6 cols → titulaciones.parquet


   ✅ recibos        :  114,454 filas × 5 cols → recibos.parquet


   ✅ domicilios     :  210,911 filas × 6 cols → domicilios.parquet


   ✅ expedientes    :  109,568 filas × 15 cols → expedientes.parquet


   ✅ demograficos   :   30,873 filas × 5 cols → demograficos.parquet


   ✅ becas          :   70,524 filas × 4 cols → becas.parquet


   ✅ trabajo        :  195,524 filas × 4 cols → trabajo.parquet


   ✅ notas          :  107,908 filas × 5 cols → notas.parquet

📖 Leyendo preinscripcion_si.xlsx...


   ✅ preinscripcion :  210,986 filas × 24 cols → preinscripcion.parquet

✅ 9 tablas procesadas y guardadas en C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim


In [4]:
# ============================================================================
# CELDA 4: GENERAR CONTENIDO HTML INTERACTIVO
# ============================================================================
#
# La página HTML de este módulo es interactiva:
#   - Muestra una rejilla de "botones" (uno por tabla)
#   - Al pasar el ratón (hover) se ven las columnas de esa tabla
#   - Al hacer clic se abre un panel con los detalles de limpieza:
#     · Transformaciones aplicadas (renombres, NaN)
#     · Tabla de columnas antes/después
#     · Métricas (filas, columnas)
#     · Enlaces a reportes Sweetviz (raw y clean)
#
# Se generan 2 bloques de HTML:
#   - botones_html: la rejilla de botones de tablas
#   - paneles_html: los paneles de detalle (uno por tabla, ocultos)
#
# El JavaScript (en la celda siguiente) se encarga de mostrar/ocultar
# los paneles cuando el usuario hace clic.
# ============================================================================

# Emoji identificativo para cada tabla (puramente estético)
EMOJIS = {
    'titulaciones': '🎓', 'recibos': '🧾', 'domicilios': '🏠',
    'expedientes': '📚', 'demograficos': '👤', 'becas': '💰',
    'trabajo': '💼', 'notas': '📝', 'preinscripcion': '📄'
}

# --- Generar botones de tablas ---
# Cada botón es un recuadro clicable que muestra:
#   - Emoji + nombre de la tabla
#   - Número de filas
#   - Tooltip con lista de columnas (al pasar el ratón)
botones_html = ''
for nombre, info in tablas_info.items():
    emoji = EMOJIS.get(nombre, '📊')
    # Generar lista de columnas para el tooltip
    cols_list = ''.join([f'<div class="col-item">{c}</div>' for c in info['cols_despues']])
    botones_html += f'''
    <div class="tabla-btn" onclick="mostrarDetalle('{nombre}')">
        <div class="tooltip-cols">
            <strong>{nombre.capitalize()} ({info['columnas']} cols)</strong>
            <div class="col-list">{cols_list}</div>
        </div>
        <div class="icon">{emoji}</div>
        <div class="name">{nombre.capitalize()}</div>
        <div class="info">{fmt(info['filas'])} filas</div>
    </div>
    '''

# --- Generar paneles de detalle ---
# Cada panel tiene 4 secciones:
#   1. Transformaciones aplicadas (badges: TIPO, NULO)
#   2. Tabla de columnas renombradas (antes → después)
#   3. Métricas (filas, columnas)
#   4. Enlaces a reportes Sweetviz (raw y clean)
paneles_html = ''
for nombre, info in tablas_info.items():
    emoji = EMOJIS.get(nombre, '📊')

    # --- Tabla de renombres (máximo 10 filas para no saturar) ---
    renombres_html = ''
    for i, (antes, despues) in enumerate(zip(info['cols_antes'], info['cols_despues'])):
        if i < 10:
            renombres_html += (
                f'<tr><td class="col-antes">{antes}</td>'
                f'<td class="col-despues">{despues}</td></tr>'
            )
    # Si hay más de 10 columnas, indicar cuántas faltan
    if len(info['cols_antes']) > 10:
        restantes = len(info['cols_antes']) - 10
        renombres_html += (
            f'<tr><td colspan="2" style="text-align:center;color:#718096;">'
            f'... y {restantes} más</td></tr>'
        )

    # --- Panel completo de esta tabla ---
    paneles_html += f'''
    <div class="detalle-panel" id="panel-{nombre}">
        <h3>{emoji} {nombre.capitalize()}</h3>
        <div class="detalle-seccion">
            <h4>🔧 Transformaciones Aplicadas</h4>
            <ul class="transform-list">
                <li><span class="badge badge-tipo">TIPO</span> Nombres de columnas convertidos a snake_case</li>
                <li><span class="badge badge-nulo">NULO</span> Valores "-" reemplazados por NaN</li>
            </ul>
        </div>

        <div class="detalle-seccion">
            <h4>📝 Columnas Renombradas</h4>
            <table class="tabla-renombres">
                <thead><tr><th>Antes (valenciano)</th><th>Después (español)</th></tr></thead>
                <tbody>{renombres_html}</tbody>
            </table>
        </div>

        <div class="detalle-seccion">
            <h4>📊 Resumen</h4>
            <div class="resumen-cambios">
                <div class="cambio-card"><div class="valor">{info['columnas']}</div><div class="label">Columnas</div></div>
                <div class="cambio-card"><div class="valor">{fmt(info['filas'])}</div><div class="label">Filas</div></div>
            </div>
        </div>
        <div class="detalle-seccion">
            <h4>🔗 Ver Reportes Sweetviz</h4>
            <div class="enlaces-sweetviz">
                <a href="reportes_raw/reporte_{nombre}.html" class="btn-sweetviz btn-raw">📋 Original</a>
                <a href="reportes_clean/reporte_{nombre}.html" class="btn-sweetviz btn-clean">✨ Limpio</a>
            </div>
        </div>
    </div>
    '''

print(f'✅ HTML generado: {len(tablas_info)} botones + {len(tablas_info)} paneles de detalle')

✅ HTML generado: 9 botones + 9 paneles de detalle


In [5]:
# ============================================================================
# CELDA 5: GENERAR PÁGINA HTML COMPLETA
# ============================================================================
#
# Junta todo el contenido generado en la celda anterior:
#   - Navegación por fases y módulos
#   - Rejilla de botones de tablas
#   - Paneles de detalle (ocultos hasta que se haga clic)
#   - JavaScript para la interactividad (mostrar/ocultar paneles)
#
# render_base_html() envuelve todo en la plantilla base.html con:
#   - Cabecera con logos UOC/UJI
#   - Footer con datos de autora (desde config_proyecto.py)
#   - Enlace al notebook en GitHub
# ============================================================================

print('=' * 60)
print('GENERANDO HTML')
print('=' * 60)

# --- Navegación dinámica ---
# fase_activa='fase1' resalta la Fase 1 en el menú superior
# modulo_activo='m02' resalta el módulo M02 en el submenú
nav_fases_html, nav_modulos_html = generar_html_navegacion_completa(
    fase_activa='fase1',
    modulo_activo='m02'
)

# --- Contenido principal ---
# Combina la intro, la rejilla de botones, y los paneles de detalle
contenido_html = f'''
<div class="intro">
    <h2>Limpieza de {len(tablas_info)} Tablas</h2>
    <p>Selecciona una tabla para ver las transformaciones aplicadas y acceder a los reportes Sweetviz.</p>
    <p><strong>👆 Hover</strong> sobre una tabla para ver sus columnas | <strong>🖱️ Clic</strong> para ver detalles</p>
</div>

<div class="tablas-grid">
    {botones_html}
</div>

<div class="detalle-panel active" id="panel-inicial">
    <div class="mensaje-inicial">
        <div class="icon">👆</div>
        <p>Selecciona una tabla para ver los detalles de limpieza</p>
    </div>
</div>

{paneles_html}
'''

# --- JavaScript para la interactividad ---
# mostrarDetalle(tabla): oculta todos los paneles y muestra solo el seleccionado
script_js = '''
<script>
    function mostrarDetalle(tabla) {
        // Ocultar todos los paneles
        document.querySelectorAll('.detalle-panel').forEach(p => p.classList.remove('active'));
        // Desactivar todos los botones
        document.querySelectorAll('.tabla-btn').forEach(b => b.classList.remove('active'));
        // Mostrar el panel seleccionado
        const panel = document.getElementById('panel-' + tabla);
        if (panel) panel.classList.add('active');
        // Resaltar el botón seleccionado
        event.currentTarget.classList.add('active');
    }
</script>
'''

# --- Generar página completa con plantilla base ---
html_completo = render_base_html(
    titulo='M02: Limpieza',
    icono='🧹',
    subtitulo='Fase 1: Transformación | TFM Abandono Universitario',
    nav_fases=nav_fases_html,
    nav_modulos=nav_modulos_html,
    contenido=contenido_html,
    scripts_adicionales=script_js,
    notebook_nombre='f1_m02_limpieza.ipynb',
    notebook_carpeta='fase1_transformacion'
)

# --- Guardar en disco ---
ruta_html = RUTA_FASE1 / 'm02_limpieza.html'
guardar_html(html_completo, ruta_html)

print(f'\n✅ HTML generado: {ruta_html}')

GENERANDO HTML
✅ HTML guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase1\m02_limpieza.html

✅ HTML generado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase1\m02_limpieza.html


In [6]:
# ============================================================================
# CELDA 6: RESUMEN FINAL
# ============================================================================
#
# Muestra un resumen de todo lo generado:
#   - Parquets limpios guardados
#   - HTML interactivo generado
#   - Métricas de cada tabla
# ============================================================================

total_filas = sum(info['filas'] for info in tablas_info.values())
total_cols = sum(info['columnas'] for info in tablas_info.values())

print('\n' + '=' * 60)
print('✅ F1-M02 COMPLETADO')
print('=' * 60)

print(f'\n📁 Archivos generados:')
print(f'   📦 {len(tablas_info)} parquets en {RUTA_INTERIM}')
print(f'   📄 {ruta_html}')

print(f'\n📊 Resumen:')
print(f'   Total filas:     {fmt(total_filas)}')
print(f'   Total columnas:  {total_cols}')

print(f'\n📋 Tablas procesadas:')
for nombre, info in tablas_info.items():
    print(f'   • {nombre:15s}: {fmt(info["filas"]):>10s} filas × {info["columnas"]} cols')

print(f'\n📌 Siguiente: f1_m03_reportes_clean.ipynb')
print('=' * 60)


✅ F1-M02 COMPLETADO

📁 Archivos generados:
   📦 9 parquets en C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
   📄 C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase1\m02_limpieza.html

📊 Resumen:
   Total filas:     1.050.793
   Total columnas:  74

📋 Tablas procesadas:
   • titulaciones   :         45 filas × 6 cols
   • recibos        :    114.454 filas × 5 cols
   • domicilios     :    210.911 filas × 6 cols
   • expedientes    :    109.568 filas × 15 cols
   • demograficos   :     30.873 filas × 5 cols
   • becas          :     70.524 filas × 4 cols
   • trabajo        :    195.524 filas × 4 cols
   • notas          :    107.908 filas × 5 cols
   • preinscripcion :    210.986 filas × 24 cols

📌 Siguiente: f1_m03_reportes_clean.ipynb
